Step 1: Import Libraries

In [ ]:
!pip install tensorflow

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, Flatten
from tensorflow.keras.utils import to_categorical

Step 2: Load Data

In [ ]:
#STEP 2A

url = "url_for_MNIST_Dataset"

#-------------------------ONLY EDIT BELOW THIS LINE ----------------------------

#TODO: Use the pandas function that reads csv files and assign it to the variable data
data = pd.read_csv(url)

# TODO: Display the first 5 rows of the dataset
print(data.head())

In [ ]:
#STEP 2B

# TODO: Assign the correct index that will display the number 4 based on the result from Step 2A
image_index = 3

# TODO: Get the pixel values for the selected image as a NumPy array
image_values = data.iloc[image_index, 1:].values

# TODO: Reshape the 1D array (784 values) into a 28x28 2D array
image_matrix = image_values.reshape(28, 28)

''' ------------------ PLEASE DO NOT EDIT BELOW THIS LINE ------------------ '''

#Display the image using matplotlib
plt.imshow(image_matrix, cmap='gray') # Use 'gray' colormap for grayscale images
plt.title(f"Label: {data.iloc[image_index, 0]}") # Add the corresponding label as a title
plt.show() # Actually display the plot

In [ ]:
#STEP 2C

# TODO: Print the dimensionality of the data
print("Shape of train_data:", data.shape)

# TODO: For all rows, select all columns from index 1 onwards
pixel_data = data.iloc[:, 1:]

# TODO: For all rows, select the first column (index 0)
labels = data.iloc[:, 0]

# TODO: Print the dimensionality of the pixel_data
print("Shape of pixel_data after separating features:", pixel_data.shape)

Step 3: Preprocess Data

In [ ]:
# TODO: Use a pandas function to convert all pixel values to numeric format
pixel_data = pixel_data.apply(pd.to_numeric, errors='coerce')

# TODO: Replace any missing values in pixel_data with 0
pixel_data = pixel_data.fillna(0)

# TODO: Normalize all pixel values to the range[0, 1]. This helps the model learn faster.
pixel_data = pixel_data.values / 255.0

# TODO: Reshape the data to include a channel dimension making it compatible with neural networks
pixel_data = pixel_data.reshape(-1, 28, 28, 1)

# TODO: Print the new dimensionality of pixel_data
print("Shape of pixel_data after reshaping:", pixel_data.shape)

Step 4: One Hot Encode Labels

In [ ]:
# TODO: One hot encode the labels and assign the correct number of classes
labels = to_categorical(labels, num_classes = 10)

# TODO: Print the new dimensionality of labels
print("Shape of labels after one-hot encoding:", labels.shape)

Step 5: Split the Data


In [ ]:
# TODO: Use sklearn function to create training and validation data splits for pixel_data and labels
# TODO: Set test size to 0.2 and random_state to 42
pixels_train, pixels_val, labels_train, labels_val = train_test_split(pixel_data, labels, test_size = 0.2, random_state = 42)
print("pixels_train shape:", pixels_train.shape)

Step 6: Build Neural Network Model

In [ ]:
# TODO: Fill in the correct structure of the model
model = Sequential([
    # Input Layer
    Input(shape=(28, 28, 1)),
    # Convert image to vector
    Flatten(),
    # Hidden layers
    Dense (128, activation='relu'),
    Dense (64, activation='relu'),
    # Output layers
    Dense(10, activation='softmax')
])

# TODO: Assign correct values for the blank parameters below
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])


''' ------------------ PLEASE DO NOT EDIT BELOW THIS LINE ------------------ '''

model.summary()

Step 7: Train the Model

In [ ]:
# TODO: Using our datasets from Step 5, train the model for 10 epochs, with a batch size of 32
history = model.fit(pixels_train, labels_train, epochs = 10, batch_size = 32, validation_data =(pixels_val, labels_val))

Step 8: Evaluate the Model


In [ ]:
# TODO: Evaluate the model using our validation sets
val_loss, val_accuracy = model.evaluate(pixels_val, labels_val)

''' ------------------ PLEASE DO NOT EDIT BELOW THIS LINE ------------------ '''

print(f"Validation Accuracy: {val_accuracy * 100:.2f}%")
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.legend()
plt.show()

Step 9: Make Predictions

In [ ]:
#TODO: Load testing data from csv
test_data = pd.read_csv('./Downloads/test.csv')

#TODO: Normalize test data values
X_test = test_data.values / 255.0

#TODO: Reshape test values to be compatible for a neural network
X_test = X_test.reshape(-1, 28, 28, 1)

# TODO: Have the model make predictions on our new data
predictions = model.predict(X_test)

# TODO: Retrieve the predictions by getting the label with the highest prediction score
predicted_labels = np.argmax(predictions, axis = 1)


''' ------------------ PLEASE DO NOT EDIT BELOW THIS LINE ------------------ '''

for i in range(5):
    plt.imshow(X_test[i].reshape(28, 28), cmap='gray')
    plt.title(f"Predicted: {predicted_labels[i]}")
    plt.axis('off')
    plt.show()

Step 10: Try It Yourself

In [ ]:
''' ------------------ PLEASE DO NOT EDIT THIS CELL ------------------ '''


from google.colab import output
import numpy as np
import matplotlib.pyplot as plt
import base64
from PIL import Image
import io

def predict_digit(data_url):
    try:
        image_data = data_url.split(",")[1]
        image = Image.open(io.BytesIO(base64.b64decode(image_data))).convert("L")

        img_array = np.array(image)

        # Find bounding box
        coords = np.column_stack(np.where(img_array > 30))
        if coords.size == 0:
            print("Draw something first!")
            return

        y_min, x_min = coords.min(axis=0)
        y_max, x_max = coords.max(axis=0)
        pad = 10
        y_min = max(0, y_min - pad)
        x_min = max(0, x_min - pad)
        y_max = min(img_array.shape[0], y_max + pad)
        x_max = min(img_array.shape[1], x_max + pad)
        img_array = img_array[y_min:y_max, x_min:x_max]

        # Resize to 20x20 (MNIST style)
        image = Image.fromarray(img_array)
        #image = image.resize((20, 20))
        image.thumbnail((20, 20), Image.LANCZOS)  # thumbnail resizes in place, keeps ratio

        # Create 28x28 canvas and center digit
        new_img = Image.new("L", (28, 28))
        # Center it properly
        x_offset = (28 - image.width) // 2
        y_offset = (28 - image.height) // 2

        new_img.paste(image, (x_offset, y_offset))

        img_array = np.array(new_img)

        # Normalize
        img_array = img_array / 255.0
        img_array = img_array.reshape(1, 28, 28, 1)

        prediction = model.predict(img_array)
        predicted_digit = np.argmax(prediction)
        confidence = np.max(prediction)

        plt.imshow(img_array.reshape(28,28), cmap='gray')
        plt.title("What the Model Sees")
        plt.show()

        print(f"Predicted Digit: {predicted_digit}")
        print(f"Confidence: {confidence:.2%}")

    except Exception as e:
        print("Error:", e)

output.register_callback('notebook.predict_digit', predict_digit)

In [ ]:
''' ------------------ PLEASE DO NOT EDIT THIS CELL ------------------ '''


from IPython.display import display, HTML
import numpy as np
import base64
from PIL import Image
import io

canvas_html = """
<canvas id="canvas" width="280" height="280"
style="border:2px solid black; background:black;"></canvas>
<br>
<button onclick="clearCanvas()">Clear</button>
<button onclick="sendImage()">Predict</button>

<script>
var canvas = document.getElementById('canvas');
var ctx = canvas.getContext('2d');
ctx.strokeStyle = "white";
ctx.lineWidth = 20;
ctx.lineJoin = "round";
ctx.lineCap = "round";

var drawing = false;

canvas.onmousedown = function(e) {
    drawing = true;
    ctx.beginPath();
    ctx.moveTo(e.offsetX, e.offsetY);
}

canvas.onmousemove = function(e) {
    if (drawing) {
        ctx.lineTo(e.offsetX, e.offsetY);
        ctx.stroke();
    }
}

canvas.onmouseup = function() {
    drawing = false;
}

function clearCanvas() {
    ctx.fillStyle = "black";
    ctx.fillRect(0, 0, canvas.width, canvas.height);
}

function sendImage() {
    var dataURL = canvas.toDataURL();
    google.colab.kernel.invokeFunction('notebook.predict_digit', [dataURL], {});
}

clearCanvas();
</script>
"""

display(HTML(canvas_html))